# ZynNova ZynMorph — TetGen + COMSOL 新功能完整测试

这个 notebook 专门验证当前 ZynMorph 的两条新生产路径：

1. **COMSOL Hex8 Mesh-v4**：验证 COMSOL tensor-product Hex8 局部节点顺序、共享面两侧性、完整域单元导出，以及“不写体单元”的 surface-only 诊断导出。
2. **TetGen 1.6 C++ 自适应四面体网格**：对一个复杂六相多孔正极生成全局共形 PLC，修复非流形体素交线，使用分相尺寸控制和局部细化区生成非均匀 Tet4，并导出 MPHTXT/VTK/MSH/INP。

默认要求已经通过：

```powershell
python scripts/vendor_tetgen.py --accept-agpl
python -m pip install -e ".[zynmorph-tetgen]" -v
```

如果只是做源码 CI/Notebook 语法检查，可以在启动 Jupyter 前设置 `ZYNNOVA_NOTEBOOK_REQUIRE_TETGEN=0`，此时 TetGen 原生网格单元会跳过；**正式验收不要关闭这个门禁**。


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import binary_dilation, gaussian_filter

try:
    import zynnova
except ModuleNotFoundError:
    candidates = [Path.cwd(), *Path.cwd().parents]
    project_root = next(
        (p for p in candidates if (p / "src" / "zynnova").is_dir()),
        None,
    )
    if project_root is None:
        raise RuntimeError(
            "没有找到 ZynNova。请在仓库根目录执行 pip install -e \".[zynmorph-tetgen]\"。"
        )
    sys.path.insert(0, str(project_root / "src"))
    import zynnova

from zynnova.geometry import tetrahedron_signed_volumes
from zynnova.zynmorph import (
    BatteryPhase,
    LocalRefinementZone,
    MicrostructureVolume,
    TetGenMeshingConfig,
    audit_comsol_hex8_topology,
    audit_multiphase_plc,
    count_nonmanifold_voxel_edges,
    export_fem_mesh,
    export_voxel_comsol_mphtxt,
    extract_multiphase_plc,
    inspect_comsol_mphtxt,
    mesh_microstructure,
    regularize_nonmanifold_junctions,
    smooth_multiphase_plc,
    tetgen_native_status,
    tetgen_native_diagnostics,
)

print("ZynNova:", zynnova.__version__)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)


## 1. 测试参数

默认体素尺寸为 150 nm，复杂测试结构为 `(z, y, x) = (16, 24, 30)`。这个规模足够产生多材料界面和非规则四面体，同时不会把 notebook 变成超大规模压力测试。


In [ ]:
SEED = 20260818
RNG = np.random.default_rng(SEED)

REQUIRE_NATIVE_TETGEN = os.environ.get("ZYNNOVA_NOTEBOOK_REQUIRE_TETGEN", "1") != "0"

OUTPUT_ROOT = Path("zynnova_runs/zynmorph_tetgen_comsol_test")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SHAPE_ZYX = (16, 24, 30)
VOXEL_SIZE_M_ZYX = (150e-9, 150e-9, 150e-9)

print("REQUIRE_NATIVE_TETGEN =", REQUIRE_NATIVE_TETGEN)
print("Output =", OUTPUT_ROOT.resolve())


## 2. 原生 TetGen / C++ 预检

这里首先确认 pip 构建出的 `_zynmorph_tetgen_native` 是否真实可导入。正式测试中不可用会立即失败，避免把规则六四面体路径误当作 TetGen。


In [ ]:
status = tetgen_native_status()
print(json.dumps({
    "available": status.available,
    "version": status.version,
    "reason": status.reason,
    "module_path": None if status.module_path is None else str(status.module_path),
    "vendored_source_path": None if status.vendored_source_path is None else str(status.vendored_source_path),
    "license": status.license,
}, indent=2, ensure_ascii=False))

if not status.available:
    diagnostics = dict(tetgen_native_diagnostics())
    print("\n=== TetGen native diagnostics ===")
    print(json.dumps(diagnostics, indent=2, ensure_ascii=False, default=str))

if REQUIRE_NATIVE_TETGEN and not status.available:
    raise RuntimeError(
        "TetGen 原生扩展不可用。上面的 diagnostics 已列出真实导入错误、zynnova/__path__、"
        "_native 搜索路径、候选 .pyd 以及 Windows DLL 搜索目录。"
        f"\n原始原因: {status.reason}\n"
        "覆盖 Native Runtime Fix 后删除 build/cp312-cp312-win_amd64，重新执行 "
        "python -m pip install -e \".[zynmorph-tetgen]\" -v，然后重启 Jupyter kernel。"
    )


## 3. COMSOL Hex8 共享面拓扑回归

使用之前问题中对应的纳米尺度间距：

```text
(2.25e-07, 4.5e-07, 2.25e-07)
```

必须满足：正 Jacobian、共享面两侧性正确、无过连接面、共享面数量完整。


In [ ]:
reported_spacing_xyz = (2.25e-7, 4.5e-7, 2.25e-7)
hex_audit = audit_comsol_hex8_topology(
    (3, 3, 3),
    spacing=reported_spacing_xyz,
    origin=(0.0, 0.0, 0.0),
)
print(json.dumps(asdict(hex_audit), indent=2))

assert hex_audit.valid
assert hex_audit.nonpositive_jacobians == 0
assert hex_audit.nonplanar_faces == 0
assert hex_audit.same_side_shared_faces == 0
assert hex_audit.overconnected_faces == 0
assert hex_audit.missing_shared_faces == 0


### 3.1 检查本机 native voxel Hex8 convention

当前 C++ backend 应报告 `comsol-v4-tensor-1`。如果 native voxel 扩展不存在，则 COMSOL Python 写入路径仍然可以测试，但正式 native 回归需要重新安装扩展。


In [ ]:
try:
    from zynnova._native import _zynsim_voxel_native as voxel_native
except Exception as exc:
    voxel_native = None
    print("Native voxel extension unavailable:", type(exc).__name__, exc)
else:
    convention = voxel_native.hex_connectivity_convention()
    print("native HEX8 convention:", convention)
    assert convention == "comsol-v4-tensor-1"


## 4. 完整 Hex8 与 surface-only MPHTXT 双重诊断

这里用一个小型三相体素场同时导出：

- `full-volume`：包含 Hex8 体单元和域 entity index；
- `surface-only`：不包含任何体单元，只保留外表面和材料界面。

二者都要能被轻量解析器重新读取。


In [ ]:
small_labels = np.zeros((3, 3, 3), dtype=np.int32)
small_labels[:, :, :1] = int(BatteryPhase.POSITIVE_ACTIVE)
small_labels[:, :, 1:2] = int(BatteryPhase.POSITIVE_ELECTROLYTE)
small_labels[:, :, 2:] = int(BatteryPhase.POSITIVE_CBD)

small_volume = MicrostructureVolume(
    labels=small_labels,
    voxel_size_m=(reported_spacing_xyz[2], reported_spacing_xyz[1], reported_spacing_xyz[0]),
    phase_names={
        int(BatteryPhase.POSITIVE_ACTIVE): "positive_active",
        int(BatteryPhase.POSITIVE_ELECTROLYTE): "positive_electrolyte",
        int(BatteryPhase.POSITIVE_CBD): "positive_cbd",
    },
)

full_hex_path = OUTPUT_ROOT / "hex8_full_volume.mphtxt"
surface_only_path = OUTPUT_ROOT / "hex8_surface_only.mphtxt"

full_hex = export_voxel_comsol_mphtxt(
    full_hex_path,
    small_volume,
    element_type="hex8",
    include_exterior_boundaries=True,
    include_material_interfaces=True,
    include_domain_entity_indices=True,
    include_volume_elements=True,
    prefer_native=True,
    validate_topology=True,
    verify=True,
)

surface_only = export_voxel_comsol_mphtxt(
    surface_only_path,
    small_volume,
    element_type="hex8",
    include_exterior_boundaries=True,
    include_material_interfaces=True,
    include_domain_entity_indices=False,
    include_volume_elements=False,
    prefer_native=False,
    validate_topology=True,
    verify=True,
)

full_info = inspect_comsol_mphtxt(full_hex_path)
surface_info = inspect_comsol_mphtxt(surface_only_path)

print("FULL:", full_hex.diagnostic_mode, full_info.element_counts)
print("SURFACE ONLY:", surface_only.diagnostic_mode, surface_info.element_counts)

assert full_hex.diagnostic_mode == "full-volume"
assert full_info.element_counts.get("hex") == small_labels.size
assert surface_only.diagnostic_mode == "surface-only"
assert surface_info.element_counts.get("hex", 0) == 0
assert surface_info.element_counts.get("quad", 0) > 0


## 5. 构造复杂六相多孔正极

相包括：

- 正极活性材料；
- 正极电解液孔隙；
- CBD；
- CEI；
- 裂纹/内部孔洞；
- 正极集流体。

活性颗粒由随机椭球重叠形成，CEI 和 CBD 使用相关随机场选择，裂纹使用斜平面与随机场交集，因此几何不会退化成规则块状结构。


In [ ]:
def select_ranked(mask: np.ndarray, score: np.ndarray, fraction: float) -> np.ndarray:
    idx = np.flatnonzero(mask)
    out = np.zeros(mask.size, dtype=bool)
    if idx.size == 0:
        return out.reshape(mask.shape)
    n = max(1, min(idx.size, int(round(fraction * idx.size))))
    if n == idx.size:
        chosen = idx
    else:
        chosen = idx[np.argpartition(score.ravel()[idx], -n)[-n:]]
    out[chosen] = True
    return out.reshape(mask.shape)


def build_complex_positive_electrode(
    shape_zyx: tuple[int, int, int],
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    labels = np.full(shape_zyx, int(BatteryPhase.POSITIVE_ELECTROLYTE), dtype=np.int32)
    zz, yy, xx = np.indices(shape_zyx, dtype=float)

    active = np.zeros(shape_zyx, dtype=bool)
    for _ in range(18):
        cz = rng.uniform(2.0, shape_zyx[0] - 3.0)
        cy = rng.uniform(3.0, shape_zyx[1] - 4.0)
        cx = rng.uniform(3.0, shape_zyx[2] - 6.0)
        rz = rng.uniform(1.5, 3.2)
        ry = rng.uniform(1.8, 4.0)
        rx = rng.uniform(2.0, 4.5)
        ellipsoid = (
            ((zz - cz) / rz) ** 2
            + ((yy - cy) / ry) ** 2
            + ((xx - cx) / rx) ** 2
            <= 1.0
        )
        active |= ellipsoid
    labels[active] = int(BatteryPhase.POSITIVE_ACTIVE)

    # 裂纹：斜平面 + 相关噪声，只切入活性材料。
    noise = gaussian_filter(rng.normal(size=shape_zyx), sigma=(1.0, 1.2, 1.5))
    plane = np.abs(xx / shape_zyx[2] + 0.33 * yy / shape_zyx[1] - 0.68) < 0.022
    threshold = np.quantile(noise[active], 0.30)
    crack = active & plane & (noise > threshold)
    labels[crack] = int(BatteryPhase.CRACK)

    # 非均匀 CEI。
    active_now = labels == int(BatteryPhase.POSITIVE_ACTIVE)
    pore = labels == int(BatteryPhase.POSITIVE_ELECTROLYTE)
    shell = binary_dilation(active_now, iterations=1) & pore
    cei_score = gaussian_filter(rng.normal(size=shape_zyx), sigma=(1.1, 1.5, 1.5))
    cei = select_ranked(shell, cei_score, 0.45)
    labels[cei] = int(BatteryPhase.POSITIVE_CEI)

    # CBD：长相关随机场 + 靠近活性材料的偏置。
    pore = labels == int(BatteryPhase.POSITIVE_ELECTROLYTE)
    near_active = binary_dilation(labels == int(BatteryPhase.POSITIVE_ACTIVE), iterations=2) & pore
    cbd_score = gaussian_filter(rng.normal(size=shape_zyx), sigma=(2.5, 1.8, 2.0)) + 0.9 * near_active
    cbd = select_ranked(pore, cbd_score, 0.13)
    labels[cbd] = int(BatteryPhase.POSITIVE_CBD)

    # x-max 两层作为正极集流体。
    labels[:, :, -2:] = int(BatteryPhase.POSITIVE_CURRENT_COLLECTOR)
    return labels


labels = build_complex_positive_electrode(SHAPE_ZYX, SEED)
volume = MicrostructureVolume(
    labels=labels,
    voxel_size_m=VOXEL_SIZE_M_ZYX,
    phase_names={int(phase): phase.name.lower() for phase in BatteryPhase},
    metadata={"seed": SEED, "purpose": "tetgen-comsol-regression"},
)

values, counts = np.unique(volume.labels, return_counts=True)
print("shape_zyx =", volume.shape)
print("phases =", {int(v): int(c) for v, c in zip(values, counts, strict=True)})
assert set(map(int, values)) == {
    int(BatteryPhase.POSITIVE_ACTIVE),
    int(BatteryPhase.POSITIVE_ELECTROLYTE),
    int(BatteryPhase.POSITIVE_CBD),
    int(BatteryPhase.POSITIVE_CEI),
    int(BatteryPhase.CRACK),
    int(BatteryPhase.POSITIVE_CURRENT_COLLECTOR),
}


### 5.1 三个正交截面


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
z0, y0, x0 = (s // 2 for s in volume.shape)
axes[0].imshow(volume.labels[z0], origin="lower", interpolation="nearest")
axes[0].set_title(f"z={z0}")
axes[1].imshow(volume.labels[:, y0, :], origin="lower", interpolation="nearest", aspect="auto")
axes[1].set_title(f"y={y0}")
axes[2].imshow(volume.labels[:, :, x0], origin="lower", interpolation="nearest", aspect="auto")
axes[2].set_title(f"x={x0}")
for ax in axes:
    ax.set_xlabel("voxel")
    ax.set_ylabel("voxel")
plt.tight_layout()
plt.show()


## 6. 非流形体素交线修复与全局共形 PLC

体素多相交界处的 checkerboard/对角接触不能直接作为 TetGen PLC。这里先统计问题数，再执行保守的确定性重标记。测试结构允许最多 4% 的体素改变；实际改变比例必须低于这个预算，并且最终非流形事件为 0。


In [ ]:
ambiguous_before = count_nonmanifold_voxel_edges(volume.labels)
print("ambiguous voxel-edge events before =", ambiguous_before)

regularized, junction_report = regularize_nonmanifold_junctions(
    volume,
    maximum_changed_fraction=0.04,
    maximum_iterations=10_000,
    minimum_phase_voxels=8,
    strict=True,
)

print(json.dumps({
    "before": junction_report.ambiguous_edges_before,
    "after": junction_report.ambiguous_edges_after,
    "changed_voxels": junction_report.changed_voxels,
    "changed_fraction": junction_report.changed_fraction,
    "iterations": junction_report.iterations,
    "converged": junction_report.converged,
}, indent=2))

assert junction_report.ambiguous_edges_after == 0
assert junction_report.changed_fraction <= 0.04
assert junction_report.converged


In [ ]:
plc = extract_multiphase_plc(
    regularized,
    checkerboard_diagonals=True,
    preserve_outer_boundary=True,
    preserve_multiphase_junctions=True,
    strict=True,
)
plc_audit = audit_multiphase_plc(plc)

max_displacement = 0.35 * min(regularized.voxel_size_m)
smoothed_plc = smooth_multiphase_plc(
    plc,
    iterations=5,
    relaxation=0.30,
    taubin_mu=-0.32,
    maximum_displacement_m=max_displacement,
)
smoothed_audit = audit_multiphase_plc(smoothed_plc)

print("PLC vertices:", len(plc.vertices))
print("PLC triangles:", len(plc.triangles))
print("raw audit valid:", plc_audit.valid)
print("smoothed audit valid:", smoothed_audit.valid)
print(json.dumps(asdict(smoothed_audit), indent=2))

assert plc_audit.valid
assert smoothed_audit.valid
assert smoothed_audit.degenerate_faces == 0
assert smoothed_audit.duplicate_faces == 0
assert smoothed_audit.open_region_edges == 0
assert smoothed_audit.nonmanifold_region_edges == 0
assert smoothed_audit.orientation_conflicts == 0


## 7. TetGen C++ 自适应四面体化

尺寸场刻意设置成明显的多尺度：裂纹和 CEI 最细，CBD 次之，活性材料和孔隙较粗，集流体最粗；同时在裂纹附近增加球形局部细化区。

这一步只有 `tetgen_native_status().available=True` 时才执行。


In [ ]:
tetgen_fem = None
tetgen_elapsed = None

if status.available:
    dz, dy, dx = regularized.voxel_size_m
    nz, ny, nx = regularized.shape
    voxel_volume = dz * dy * dx

    crack_focus_xyz = (
        0.55 * nx * dx,
        0.40 * ny * dy,
        0.50 * nz * dz,
    )

    tetgen_config = TetGenMeshingConfig(
        radius_edge_ratio=1.50,
        minimum_dihedral_degrees=8.0,
        optimization_level=2,
        phase_maximum_tetra_volume_m3={
            int(BatteryPhase.POSITIVE_ACTIVE): 0.45 * voxel_volume,
            int(BatteryPhase.POSITIVE_ELECTROLYTE): 0.80 * voxel_volume,
            int(BatteryPhase.POSITIVE_CBD): 0.35 * voxel_volume,
            int(BatteryPhase.POSITIVE_CEI): 0.18 * voxel_volume,
            int(BatteryPhase.CRACK): 0.12 * voxel_volume,
            int(BatteryPhase.POSITIVE_CURRENT_COLLECTOR): 1.00 * voxel_volume,
        },
        local_refinement_zones=(
            LocalRefinementZone(
                center_m_xyz=crack_focus_xyz,
                radius_m=0.22 * min(nx * dx, ny * dy, nz * dz),
                maximum_tetra_volume_m3=0.08 * voxel_volume,
                name="crack_refinement",
            ),
        ),
        smoothing_iterations=5,
        smoothing_relaxation=0.30,
        smoothing_taubin_mu=-0.32,
        maximum_surface_displacement_voxels=0.35,
        regularize_junctions=False,  # 上一步已经显式完成并审计
        consistency_check=True,
        conforming_delaunay=True,
        quiet=True,
    )

    t0 = time.perf_counter()
    tetgen_fem = mesh_microstructure(
        regularized,
        method="tetgen",
        tetgen_config=tetgen_config,
        maximum_tetrahedra=1_500_000,
    )
    tetgen_elapsed = time.perf_counter() - t0

    print("backend:", tetgen_fem.backend)
    print("nodes:", tetgen_fem.mesh.n_nodes)
    print("tetrahedra:", tetgen_fem.mesh.n_cells)
    print("elapsed_s:", round(tetgen_elapsed, 3))
    print("quality:", tetgen_fem.quality)

    assert tetgen_fem.backend.startswith("tetgen")
    assert tetgen_fem.quality.fem_ready
    assert tetgen_fem.quality.inverted_cells == 0
    assert tetgen_fem.quality.degenerate_cells == 0
    assert tetgen_fem.mesh.n_cells != regularized.labels.size * 6
else:
    print("SKIPPED: native TetGen unavailable and REQUIRE_NATIVE_TETGEN=0")


### 7.1 验证“不是规则立方体拆分”的非均匀性

使用实际 Tet4 体积分布验证网格具有多尺度。这里看 5/50/95 分位数和 `q95/q05`，而不是只看单元总数。


In [ ]:
volume_stats = None
if tetgen_fem is not None:
    signed_volumes = tetrahedron_signed_volumes(tetgen_fem.mesh)
    positive_volumes = signed_volumes[signed_volumes > 0]
    q05, q50, q95 = np.quantile(positive_volumes, [0.05, 0.50, 0.95])
    volume_stats = {
        "q05_m3": float(q05),
        "q50_m3": float(q50),
        "q95_m3": float(q95),
        "q95_over_q05": float(q95 / q05),
        "coefficient_of_variation": float(np.std(positive_volumes) / np.mean(positive_volumes)),
    }
    print(json.dumps(volume_stats, indent=2))

    # 生产目标是多尺度而不是所有 Tet4 等体积。
    assert volume_stats["q95_over_q05"] > 1.10
    assert volume_stats["coefficient_of_variation"] > 0.03

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.hist(np.log10(positive_volumes), bins=50)
    ax.set_xlabel("log10(Tet volume / m³)")
    ax.set_ylabel("count")
    ax.set_title("Adaptive TetGen tetrahedron-volume distribution")
    plt.tight_layout()
    plt.show()


## 8. TetGen 网格导出：COMSOL / VTK / Gmsh / Abaqus

TetGen 网格通过 ZynMorph 的统一 `export_fem_mesh` 输出，并重新读取 MPHTXT 进行节点/单元计数核对。


In [ ]:
exported = None
tet_info = None

if tetgen_fem is not None:
    mesh_dir = OUTPUT_ROOT / "adaptive_tetgen_mesh"
    exported = export_fem_mesh(
        tetgen_fem,
        mesh_dir,
        formats=("vtk", "msh", "inp", "mphtxt"),
        export_boundary=True,
        comsol_domain_selections={
            "active_material": (int(BatteryPhase.POSITIVE_ACTIVE),),
            "electrolyte": (int(BatteryPhase.POSITIVE_ELECTROLYTE),),
            "electronic_network": (
                int(BatteryPhase.POSITIVE_CBD),
                int(BatteryPhase.POSITIVE_CURRENT_COLLECTOR),
            ),
            "interphase": (int(BatteryPhase.POSITIVE_CEI),),
            "damage": (int(BatteryPhase.CRACK),),
        },
        comsol_options={
            "include_boundaries": True,
            "include_internal_interfaces": True,
            "include_exterior": True,
            "create_interface_selections": True,
            "verify": True,
        },
    )

    tet_info = inspect_comsol_mphtxt(exported.exports["mphtxt"])
    print("TetGen MPHTXT:", exported.exports["mphtxt"].resolve())
    print("MPHTXT element counts:", tet_info.element_counts)
    print("exports:")
    for key, path in sorted(exported.exports.items()):
        print(f"  {key:14s} {path}  {path.stat().st_size / 2**20:.3f} MiB")

    assert tet_info.element_counts.get("tet") == tetgen_fem.mesh.n_cells
    assert tet_info.element_counts.get("tri", 0) > 0
    assert exported.exports["vtk"].is_file()
    assert exported.exports["msh"].is_file()
    assert exported.exports["inp"].is_file()
    assert exported.exports["mphtxt"].is_file()


## 9. 最终机器可读验收报告

正式环境中最终报告必须同时显示 Hex8 拓扑通过、PLC 通过、TetGen native 可用、Tet4 FEM-ready，并且自适应单元体积具有明显分布宽度。


In [ ]:
summary = {
    "schema": "zynnova.zynmorph.tetgen-comsol-notebook.v1",
    "zynnova_version": zynnova.__version__,
    "seed": SEED,
    "native_tetgen_required": REQUIRE_NATIVE_TETGEN,
    "native_tetgen_available": status.available,
    "native_tetgen_version": status.version,
    "hex8_audit": asdict(hex_audit),
    "hex8_full_export": {
        "path": str(full_hex_path.resolve()),
        "diagnostic_mode": full_hex.diagnostic_mode,
        "element_counts": dict(full_info.element_counts),
    },
    "hex8_surface_only_export": {
        "path": str(surface_only_path.resolve()),
        "diagnostic_mode": surface_only.diagnostic_mode,
        "element_counts": dict(surface_info.element_counts),
    },
    "microstructure": {
        "shape_zyx": list(regularized.shape),
        "phases": list(map(int, regularized.phases)),
        "ambiguous_edges_before": junction_report.ambiguous_edges_before,
        "ambiguous_edges_after": junction_report.ambiguous_edges_after,
        "changed_fraction": junction_report.changed_fraction,
    },
    "plc": asdict(smoothed_audit),
    "tetgen": None,
}

if tetgen_fem is not None:
    summary["tetgen"] = {
        "backend": tetgen_fem.backend,
        "elapsed_s": tetgen_elapsed,
        "nodes": tetgen_fem.mesh.n_nodes,
        "tetrahedra": tetgen_fem.mesh.n_cells,
        "quality": asdict(tetgen_fem.quality),
        "volume_distribution": volume_stats,
        "mphtxt": None if exported is None else str(exported.exports["mphtxt"].resolve()),
    }

report_path = OUTPUT_ROOT / "validation_summary.json"
report_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nValidation report:", report_path.resolve())

assert summary["hex8_audit"]["valid"]
assert summary["plc"]["valid"]
if REQUIRE_NATIVE_TETGEN:
    assert summary["native_tetgen_available"]
    assert summary["tetgen"] is not None
    assert summary["tetgen"]["quality"]["fem_ready"]

print("\n=== ALL REQUESTED GATES PASSED ===")


## 10. COMSOL 最终人工导入检查

自动测试能够验证 MPHTXT 结构、节点/单元数量、Hex8 Jacobian、共享面两侧性、Tet4 方向和几何实体记录，但最终生产验收仍建议在 COMSOL 中实际执行一次导入：

1. 导入 `hex8_full_volume.mphtxt`，确认不再出现“两个单元连接到共享单元面的同一侧”。
2. 如仍有环境特异问题，导入 `hex8_surface_only.mphtxt` 区分体拓扑与二维界面问题。
3. 导入 `adaptive_tetgen_mesh/microstructure.mphtxt`，检查材料域与 interface selections。
4. 在 COMSOL Mesh statistics 中检查 inverted/invalid element 为 0。
5. 对关键 SEI/CEI/裂纹区域检查局部 Tet4 尺寸确实更细。
